# Arize Singapore Workshop - the dev loop for agents

Build a **LangGraph customer-support agent** for a fictional outdoor retailer
(Sunrise Outfitters), then take it through the core Arize AX loop:

**build -> trace -> offline-eval -> experiment -> chat live & watch traces.**

The agent does **real retrieval** over a small policy knowledge base and
**escalates** hard or high-risk requests to a human - the most common
enterprise customer-support pattern.

### Agenda (~50-60 min hands-on)
| # | Section | ~Time |
|---|---------|------|
| 1 | Setup & keys | 5 min |
| 2 | Build the agent (retrieval + escalation), run untraced | 10 min |
| 3 | Add Arize tracing, explore traces | 10 min |
| 4 | Offline evals against a golden dataset | 10-15 min |
| 5 | Experiment: compare two models | 10 min |
| 6 | Live chat via Gradio (everyone generates traces) | 5-10 min |
| 7 | (Optional / take-home) single -> multi-agent | - |

**You only need three keys:** a Google AI Studio API key, and your Arize Space ID +
API key (app.arize.com -> Settings -> Space API Keys).

## 1. Setup & keys

Fetch the repo (so Colab has the `agent/` and `evals/` code), install
dependencies, and enter your keys. We import the **agent** from the repo's
`agent/` package, and write the **evaluation code inline** in this notebook so
you can see every Arize SDK call as you run it.

In [ ]:
# Clone the workshop repo into the Colab VM and install dependencies.
!git clone -q https://github.com/FSurani/arize-singapore-workshop-2026.git
%cd arize-singapore-workshop-2026
%pip install -q -r requirements.txt

In [ ]:
import os
from getpass import getpass

os.environ['GOOGLE_API_KEY'] = getpass('Google AI Studio API key: ')
os.environ['ARIZE_SPACE_ID'] = getpass('Arize Space ID: ')
os.environ['ARIZE_API_KEY'] = getpass('Arize API key: ')
os.environ['ARIZE_PROJECT'] = 'arize-singapore-workshop'

## 2. Build the support agent (retrieval + escalation)

The agent is a LangGraph ReAct agent with four tools:
- `lookup_order` - mock order-management API (orders A1001-A1003)
- `check_refund_eligibility` - applies the return policy
- `search_knowledge_base` - **real retrieval** over policy docs (shipping, returns, sizing, warranty, payments)
- `escalate_to_human` - opens a ticket for upset / out-of-scope / high-risk requests

Let's build it, run a couple of queries, and try it in a chat UI - all with
**no tracing yet** (we add tracing in Section 3, then bring the UI back in
Section 6 to generate real traces).

In [ ]:
from agent.graph import build_agent, run_agent

agent = build_agent()
print(run_agent(agent, 'How long does shipping to Singapore take?'))  # uses retrieval
print('---')
print(run_agent(agent, 'Where is my order A1001?'))                   # uses order lookup

In [ ]:
# Chat with the agent in a UI right now. Ask about orders A1001-A1003, shipping,
# returns, sizing, or try an angry message. (Not traced yet - that's Section 3.)
from app import build_demo

build_demo(agent).launch(share=True)

## 3. Add Arize tracing and explore

Tracing is just **two lines**: `register(...)` then
`LangChainInstrumentor().instrument(...)`. Because LangGraph runs on LangChain
runnables, that single instrumentor captures the whole graph - agent reasoning,
every LLM call, and every tool call (including the **retrieval** span with the
passages it pulled). No changes to the agent code itself.

In [ ]:
from arize.otel import register
from openinference.instrumentation.langchain import LangChainInstrumentor

# This is the whole tracing setup: register a tracer, then auto-instrument
# LangChain. That single instrumentor captures the entire LangGraph agent -
# its reasoning, every LLM call, and every tool/retriever call.
tracer_provider = register(
    space_id=os.environ['ARIZE_SPACE_ID'],
    api_key=os.environ['ARIZE_API_KEY'],
    project_name=os.environ['ARIZE_PROJECT'],
)
LangChainInstrumentor().instrument(tracer_provider=tracer_provider)

# Rebuild the agent so it runs under the tracer.
agent = build_agent()

In [ ]:
for q in [
    'What is your return policy?',
    'Can I get a refund on order A1002?',
    'Order A1003 still has not arrived and I am furious!',  # should escalate
]:
    print('Q:', q)
    print('A:', run_agent(agent, q))
    print('-' * 60)

# Tracing uses a BatchSpanProcessor (spans are buffered and sent on a timer).
# Force a flush so all traces above are pushed to Arize right now.
tracer_provider.force_flush()
print(f"Open the '{os.environ['ARIZE_PROJECT']}' project in Arize to see the traces "
      "(look for the nested RETRIEVER span under the policy question's tool call).")

## 4. Offline evals against a golden dataset

We ship a small hand-labeled dataset (`evals/datasets/golden_support.json`,
~16 examples incl. edge cases and 'should escalate' rows). We'll:

1. **Create a dataset** in Arize from it (`client.datasets.create`).
2. **Define a task + 2 evaluators** (inline, so you see exactly what's scored).
3. **Run an experiment** (`client.experiments.run`) and read per-example scores.

The two evaluators:
- **tool_selection** (code) - did it call the expected tool?
- **groundedness** (LLM judge) - is the reply supported by retrieved context?

Two more (**correctness** and **escalation_appropriate**) are left as exercises
for you - see the TODO in the next cell.

In [ ]:
import json
from arize import ArizeClient

client = ArizeClient(api_key=os.environ['ARIZE_API_KEY'])
SPACE = os.environ['ARIZE_SPACE_ID']
DATASET = 'sunrise-support-golden'

# Hand-labeled golden examples ship with the repo.
examples = json.load(open('evals/datasets/golden_support.json'))
print(f'{len(examples)} examples. First one:')
print(json.dumps(examples[0], indent=2))

# Create the dataset in Arize. Safe to re-run: we fall back to the existing one.
try:
    dataset = client.datasets.create(name=DATASET, space=SPACE, examples=examples)
    print('Created dataset:', dataset.id)
except Exception as e:
    print('Dataset may already exist; will reuse it by name.', str(e)[:120])

In [ ]:
# Define the task (run the agent) and the evaluators - inline so you can see
# exactly what each one scores. Arize calls these for every dataset row.
from arize.experiments import EvaluationResult
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from agent.graph import message_text

RETRIEVAL_TOOL, ESCALATION_TOOL = 'search_knowledge_base', 'escalate_to_human'


def agent_response(agent, message):
    """Run the agent once; capture the reply, tools used, and retrieved context."""
    msgs = agent.invoke({'messages': [HumanMessage(content=message)]})['messages']
    tools_used, context = [], []
    for m in msgs:
        if isinstance(m, AIMessage):
            for call in getattr(m, 'tool_calls', None) or []:
                tools_used.append(call['name'])
        elif isinstance(m, ToolMessage) and m.name == RETRIEVAL_TOOL:
            context.append(str(m.content))
    reply = ''
    for m in reversed(msgs):
        if isinstance(m, AIMessage) and m.content:
            reply = message_text(m.content)
            if reply:
                break
    return {'reply': reply, 'tools_used': tools_used, 'context': '\n\n'.join(context)}


def make_task(agent):
    def task(row):
        return json.dumps(agent_response(agent, row.get('input', '')))
    return task


def _parse(out):
    try:
        return json.loads(out)
    except Exception:
        return {'reply': str(out), 'tools_used': [], 'context': ''}


# --- Evaluator 1: code-based (tool selection) ---
def tool_selection(output, dataset_row):
    expected = dataset_row.get('expected_tool')
    used = _parse(output).get('tools_used', [])
    if not expected:
        return EvaluationResult(label='not_applicable', explanation='No specific tool expected.')
    ok = expected in used
    return EvaluationResult(score=int(ok), label='correct' if ok else 'incorrect',
                            explanation=f"Expected '{expected}'; agent used {used or 'no tools'}.")


# --- Evaluator 2: LLM-as-a-judge (groundedness) ---
def _judge(system, user):
    llm = ChatGoogleGenerativeAI(model='gemini-3.1-flash-lite', temperature=0)
    text = message_text(llm.invoke(
        [SystemMessage(content=system), HumanMessage(content=user)]).content).strip()
    return (text.splitlines()[0].lower() if text else ''), text


def groundedness(output, dataset_row):
    d = _parse(output)
    ctx = d.get('context', '')
    expects = dataset_row.get('expected_tool') == RETRIEVAL_TOOL
    if not ctx:
        if expects:  # policy question answered without retrieving = ungrounded
            return EvaluationResult(score=0, label='ungrounded',
                                    explanation='Policy question answered without retrieval.')
        return EvaluationResult(label='not_applicable', explanation='Retrieval not expected for this row.')
    label, expl = _judge(
        "Is the reply grounded in the retrieved context? "
        "First line: 'grounded' or 'hallucinated', then one sentence why.",
        f"Context:\n{ctx}\n\nReply:\n{d.get('reply','')}")
    ok = 'grounded' in label and 'hallucinated' not in label
    return EvaluationResult(score=int(ok), label='grounded' if ok else 'hallucinated', explanation=expl)


# ---------------------------------------------------------------------------
# TODO (exercise): implement two more evaluators, then add them to EVALUATORS.
#   1. correctness (LLM judge): does `reply` match dataset_row['expected_output']?
#      Hint: reuse _judge(...) like groundedness does.
#   2. escalation_appropriate (code): did the agent escalate iff expected?
#      Hint: compare (ESCALATION_TOOL in tools_used) with
#            bool(dataset_row.get('expect_escalation')).
# ---------------------------------------------------------------------------

EVALUATORS = [tool_selection, groundedness]
print('Task and', len(EVALUATORS), 'evaluators defined.')

In [ ]:
# Run the experiment: the agent over every dataset row, scored by the 4 evaluators.
from datetime import datetime

experiment, df = client.experiments.run(
    name='offline-baseline-' + datetime.now().strftime('%m%d-%H%M'),
    dataset=DATASET,
    space=SPACE,
    task=make_task(agent),
    evaluators=EVALUATORS,
    concurrency=4,
)
print('Logged experiment:', experiment.id if experiment else None)
print('Open the dataset in Arize -> Experiments to see per-example scores.')

## 5. Experiment: compare two models

Run the same dataset + evaluators through two different models and compare
them side by side in Arize's Experiment Comparison view:
- **gemma-4-31b-it** - smaller open model, weaker reasoning
- **gemini-3.1-flash-lite** - stronger, more capable

Expect the stronger model to score higher on groundedness / tool-selection.
That gap is exactly what experiments make visible.

In [ ]:
# Same dataset + evaluators, two different models => directly comparable in Arize.
from datetime import datetime

suffix = datetime.now().strftime('%m%d-%H%M')
for model in ['gemma-4-31b-it', 'gemini-3.1-flash-lite']:
    client.experiments.run(
        name=f'{model}-{suffix}', dataset=DATASET, space=SPACE,
        task=make_task(build_agent(model=model)),
        evaluators=EVALUATORS, concurrency=4,
    )
    print('Logged experiment for', model)

print(f"Compare 'gemma-4-31b-it-{suffix}' vs 'gemini-3.1-flash-lite-{suffix}' in Arize.")

## 6. Live chat via Gradio - everyone generates traces

Here's the same chat UI from Section 2 - but now tracing is on, so **every
conversation is traced into Arize**. Chat for a minute (orders A1001-A1003,
shipping, returns, sizing, or an angry message), then open the project in Arize
and watch your traces appear.

In [ ]:
# Same UI as Section 2, but `agent` was rebuilt under the tracer in Section 3,
# so these conversations are traced into Arize.
from app import build_demo

demo = build_demo(agent)
demo.launch(share=True)

## 7. (Optional / take-home) Single -> multi-agent

The core workshop uses a single tool-using agent - the dominant production CX
pattern. For complex workflows, the 2026 trend is multi-agent orchestration: a
router/orchestrator delegating to specialist sub-agents (e.g. an orders agent
and a returns agent). It adds token cost and debugging overhead, so it is left
as an extension.

**Take-home idea:** wrap each tool group in its own `create_react_agent`, add a
router agent that picks which specialist to call, and trace it - the single
LangChain instrumentor will capture the full multi-agent graph in Arize.

### Recap
You built a retrieval + escalation support agent, traced it with two lines,
evaluated it offline against a golden dataset, and compared two models in an
experiment - the core Arize AX dev loop.